In [1]:
import numpy as np
import torch

In [2]:
for i in range(7,-1,-1):
    print(i)

7
6
5
4
3
2
1
0


In [3]:
BATCH_SIZE = 5
N_in = 2
N_out = 3
input = np.random.randint(0, 5, (BATCH_SIZE, N_in))
W = np.random.randint(5,10,(N_out,N_in))
B = np.random.randint(1,2,(N_out))

In [4]:
B

array([1, 1, 1], dtype=int32)

In [5]:
input

array([[0, 2],
       [3, 1],
       [2, 0],
       [3, 4],
       [2, 2]], dtype=int32)

In [6]:
W

array([[8, 6],
       [5, 5],
       [8, 6]], dtype=int32)

In [7]:

np.dot(input, W.T) + B

array([[13, 11, 13],
       [31, 21, 31],
       [17, 11, 17],
       [49, 36, 49],
       [29, 21, 29]], dtype=int32)

In [8]:
s = np.subtract(input, input.max(axis=1, keepdims=True))
s = s.astype(np.float64)

In [9]:
s

array([[-2.,  0.],
       [ 0., -2.],
       [ 0., -2.],
       [-1.,  0.],
       [ 0.,  0.]])

In [10]:
input

array([[0, 2],
       [3, 1],
       [2, 0],
       [3, 4],
       [2, 2]], dtype=int32)

In [11]:
for i in range(0,s.shape[0]):
    s[i] = np.log(np.exp(s[i])/sum(np.exp(s[i])))

In [12]:
s

array([[-2.12692801, -0.12692801],
       [-0.12692801, -2.12692801],
       [-0.12692801, -2.12692801],
       [-1.31326169, -0.31326169],
       [-0.69314718, -0.69314718]])

In [13]:
np.sum(s, axis=-1, keepdims=True)

array([[-2.25385602],
       [-2.25385602],
       [-2.25385602],
       [-1.62652338],
       [-1.38629436]])

In [14]:
torch.nn.functional.log_softmax(torch.Tensor(s),dim=1)

tensor([[-2.1269, -0.1269],
        [-0.1269, -2.1269],
        [-0.1269, -2.1269],
        [-1.3133, -0.3133],
        [-0.6931, -0.6931]])

In [15]:
input = np.random.randint(2,5,(5,2))

In [16]:
input

array([[2, 4],
       [2, 3],
       [3, 3],
       [2, 3],
       [2, 4]], dtype=int32)

In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(with_mean=True, with_std=True)
std = scaler.fit_transform(input)



In [18]:
print(std)
print(input.mean(axis=1,keepdims=True))
output = np.subtract(input, input.mean(axis=0, keepdims=True))/np.sqrt(input.var(axis=0, keepdims=True) + 0.001)
print(output)

[[-0.5         1.22474487]
 [-0.5        -0.81649658]
 [ 2.         -0.81649658]
 [-0.5        -0.81649658]
 [-0.5         1.22474487]]
[[3. ]
 [2.5]
 [3. ]
 [2.5]
 [3. ]]
[[-0.49844479  1.22220127]
 [-0.49844479 -0.81480084]
 [ 1.99377915 -0.81480084]
 [-0.49844479 -0.81480084]
 [-0.49844479  1.22220127]]


In [19]:
input = np.random.randint(-5,5,(5,2))
gradOutput = np.random.randint(0,5,(5,2))


In [20]:
torch_layer = torch.nn.LeakyReLU(input)
torch_layer

LeakyReLU(
  negative_slope=[[ 3 -2]
   [-3  0]
   [ 2  3]
   [ 0  1]
   [ 3 -3]]
)

In [21]:
input[input < 0] = 0
print(input)


[[3 0]
 [0 0]
 [2 3]
 [0 1]
 [3 0]]


In [22]:
s = np.sum(gradOutput*input, axis=1, keepdims=True)

In [23]:
print(input)
torch.nn.functional.dropout1d(torch.Tensor(input))

[[3 0]
 [0 0]
 [2 3]
 [0 1]
 [3 0]]


tensor([[0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 2.],
        [0., 0.]])

In [24]:
(np.random.random(input.shape) > 0.5)/(1-0.5)

array([[2., 2.],
       [0., 2.],
       [0., 2.],
       [2., 2.],
       [0., 0.]])

In [25]:
np.random.binomial((5,2),0.5)


array([1, 1], dtype=int32)

In [26]:
from functions import ClassNLLCriterionUnstable
import numpy as np
import torch
np.random.seed(42)
torch.manual_seed(42)

batch_size, n_in = 2, 4
for _ in range(100):
    # layers initialization
    torch_layer = torch.nn.NLLLoss()
    custom_layer = ClassNLLCriterionUnstable()

    layer_input = np.random.uniform(0, 1, (batch_size, n_in)).astype(np.float32)
    layer_input /= layer_input.sum(axis=-1, keepdims=True)
    layer_input = layer_input.clip(custom_layer.EPS, 1. - custom_layer.EPS)  # unifies input
    target_labels = np.random.choice(n_in, batch_size)
    target = np.zeros((batch_size, n_in), np.float32)
    target[np.arange(batch_size), target_labels] = 1  # one-hot encoding

    # 1. check layer output
    custom_layer_output = custom_layer.updateOutput(layer_input, target)
    layer_input_var = torch.from_numpy(layer_input).float().requires_grad_(True)
    torch_layer_output_var = torch_layer(
        torch.log(layer_input_var),
        torch.from_numpy(target_labels))

RuntimeError: expected scalar type Long but found Int

In [342]:
input = np.random.randint(0,5, (5,3,6,6))
# input = np.random.randint(0,5, (6,6))


In [343]:
input

array([[[[0, 0, 0, 1, 4, 0],
         [4, 3, 1, 4, 0, 0],
         [4, 4, 3, 1, 2, 3],
         [3, 3, 0, 4, 1, 0],
         [3, 1, 0, 2, 3, 0],
         [0, 3, 2, 4, 2, 3]],

        [[4, 0, 2, 2, 4, 0],
         [2, 4, 2, 0, 0, 1],
         [4, 3, 2, 0, 2, 2],
         [3, 0, 1, 4, 2, 3],
         [3, 4, 4, 0, 2, 2],
         [4, 1, 2, 3, 4, 2]],

        [[0, 4, 3, 0, 4, 4],
         [2, 0, 4, 3, 0, 2],
         [1, 4, 1, 4, 2, 2],
         [2, 2, 0, 0, 4, 0],
         [4, 0, 0, 0, 3, 2],
         [3, 0, 0, 3, 4, 2]]],


       [[[4, 2, 1, 2, 4, 2],
         [1, 1, 0, 1, 3, 4],
         [3, 3, 4, 3, 3, 3],
         [1, 4, 4, 4, 3, 1],
         [4, 2, 4, 0, 2, 4],
         [3, 4, 1, 1, 4, 3]],

        [[3, 3, 4, 4, 0, 1],
         [2, 2, 1, 0, 1, 0],
         [0, 3, 1, 2, 4, 4],
         [1, 4, 0, 4, 0, 0],
         [2, 4, 2, 4, 0, 4],
         [2, 3, 2, 3, 1, 2]],

        [[4, 3, 4, 3, 2, 2],
         [4, 2, 4, 0, 3, 4],
         [3, 0, 3, 4, 2, 3],
         [3, 0, 1, 2, 4, 0],
  

In [362]:
def updateOutput(input):
    kernel_size = 2
    input_h, input_w = input.shape[-2:]
    max_indices = np.zeros((input.shape[0],input.shape[1],input_h//2,input_w//2))
    # your may remove these asserts and implement MaxPool2d with padding
    output = np.zeros((input.shape[0],input.shape[1],input_h//2,input_w//2))
    gradOutput = np.random.randint(0,5,(input.shape[0],input.shape[1],input_h//2, input_w//2))
    gradinput = np.zeros((input.shape[0],input.shape[1],input_h, input_w))
    # YOUR CODE #############################
    for batch in range(input.shape[0]):
        for ch in range(input.shape[1]):
            new_h = 1
            for h in range(0,input_h,kernel_size):
                new_w = 1
                for w in range(0,input_w,kernel_size):
                    window = np.array(input[batch, ch, h:h+kernel_size, w:w+kernel_size], copy=True)
                    window = window.reshape(-1)
                    window = np.zeros_like(window)
                    window[max_indices[batch, ch, new_h, new_w].astype(np.int32)] = gradOutput[batch, ch, new_h, new_w]
                    print(gradinput[batch, ch, h:h+kernel_size, w:w+kernel_size])
                    gradinput[batch, ch, h:h+kernel_size, w:w+kernel_size] = window.reshape(kernel_size, kernel_size)
                    print(gradinput[batch, ch, h:h+kernel_size, w:w+kernel_size])
                    break
                    new_w +=1
                break
                new_h += 1
            break
        break
    return max_indices
updateOutput(input)

[[0. 0.]
 [0. 0.]]
[[2. 0.]
 [0. 0.]]


array([[[[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]]],


       [[[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]]],


       [[[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]]],


       [[[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]]],


       [[[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.],
         [0., 0., 0.]],

In [135]:
s = np.zeros((2,2))
s[3]

IndexError: index 3 is out of bounds for axis 0 with size 2

In [ ]:
def adam_optimizer(variables, gradients, config, state):  
    # 'variables' and 'gradients' have complex structure, accumulated_grads will be stored in a simpler one
    state.setdefault('m', {})  # first moment vars
    state.setdefault('v', {})  # second moment vars
    state.setdefault('t', 0)   # timestamp
    state['t'] += 1
    for k in ['learning_rate', 'beta1', 'beta2', 'epsilon']:
        assert k in config, config.keys()
    
    var_index = 0 
    lr_t = config['learning_rate'] * np.sqrt(1 - config['beta2']**state['t']) / (1 - config['beta1']**state['t'])
    for current_layer_vars, current_layer_grads in zip(variables, gradients): 
        for current_var, current_grad in zip(current_layer_vars, current_layer_grads):
            var_first_moment = state['m'].setdefault(var_index, np.zeros_like(current_grad))
            var_second_moment = state['v'].setdefault(var_index, np.zeros_like(current_grad))
            
            # <YOUR CODE> #######################################
            # update `current_var_first_moment`, `var_second_moment` and `current_var` values
            #np.add(... , out=var_first_moment)
            #np.add(... , out=var_second_moment)
            #current_var -= ...

            np.add(config['beta1'] * var_first_moment,
                (1 - config['beta1']) * current_grad,
                out=var_first_moment)


            np.add(config['beta2'] * var_second_moment,
                (1 - config['beta2']) * np.square(current_grad),
                out=var_second_moment)


            m_hat = var_first_moment / (1 - config['beta1'] ** state['t'])


            v_hat = var_second_moment / (1 - config['beta2'] ** state['t'])


            current_var -= lr_t * m_hat / (np.sqrt(v_hat) + config['epsilon'])

            # small checks that you've updated the state; use np.add for rewriting np.arrays values
            assert var_first_moment is state['m'].get(var_index)
            assert var_second_moment is state['v'].get(var_index)
            var_index += 1
state = {}  
config = {'learning_rate': 1e-3, 'beta1': 0.9, 'beta2':0.999, 'epsilon':1e-8}
variables = [[np.arange(10).astype(np.float64)]]
gradients = [[np.arange(10).astype(np.float64)]]
print(adam_optimizer(variables, gradients, config, state))


None
